# 🌾 India Districts Crop Production — Full ML Pipeline

**Dataset:** India Districts Crop Production (27,303 rows × 21 columns)  
**Target:** `Production` — crop output in thousand tonnes across Indian districts  
**Years covered:** 2021-22, 2022-23, 2023-24  
**Crops:** 19 types including Rice, Wheat, Maize, Bajra, Pulses, etc.

**Pipeline:**
1. Load & EDA
2. Handle zeros & log-transform skewed features
3. Feature engineering & preprocessing (no leakage)
4. Baseline models (Linear Regression, Decision Tree, Random Forest, SVR)
5. Hyperparameter tuning (GridSearchCV / RandomizedSearchCV)
6. XGBoost & LightGBM
7. Learning curves
8. Residual analysis
9. SHAP (global + local XAI)
10. LIME (local XAI)
11. Final comparison table — all metrics

---
## 1. Imports & Package Setup

We import all required libraries upfront and auto-install any ML/XAI packages that may not already be present in the environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import subprocess, sys
for pkg in ['xgboost', 'lightgbm', 'shap', 'lime']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

print('✅ All packages ready.')

---
## 2. Load & Explore the Data

### About this dataset
Each row represents a **crop grown in a specific district in a specific year**. Features include:
- **Geographic:** State, District, Latitude, Longitude
- **Agronomic:** Crop type, Year
- **Soil:** Nitrogen, Phosphorus, Potassium (NPK), Organic Carbon, Soil pH
- **Weather:** Temperature, Humidity, Precipitation, Sunshine hours
- **Pre-encoded:** State_Encoded, District_Encoded, Crop_Encoded (label-encoded versions of categoricals)
- **Target:** `Production` (in thousand tonnes)

Note: `Country` is always 'India' — we drop it. `Ind_adm2_ID` is just a geographic ID — not a useful predictor.

In [ ]:
df = pd.read_csv('India_Districts_Crop_Production_Processed.csv')

print('Shape:', df.shape)
print('\nColumns:', list(df.columns))
df.head()

In [ ]:
# Statistical summary — check min/max/mean for anomalies
# Key things to notice:
# - Production min = 0, 25th percentile = 0 → more than half the rows are zero production!
# - Production max = 3090 but mean = 90 → extreme right skew
df.describe()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\n✅ No missing values in this dataset — no imputation needed.')

In [ ]:
# Dataset composition
print('Unique States :', df['State'].nunique())
print('Unique Districts:', df['District'].nunique())
print('Unique Crops  :', df['Crop'].nunique(), '→', list(df['Crop'].unique()))
print('Years covered :', list(df['Year'].unique()))

print(f'\nZero-production rows: {(df["Production"]==0).sum()} out of {len(df)} ({(df["Production"]==0).mean()*100:.1f}%)')
print('This means many districts did not grow a particular crop in a given year.')

### Visualising the Data

In [ ]:
# Top 10 crops by total production
top_crops = df.groupby('Crop')['Production'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 5))
top_crops.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Top 10 Crops by Total Production (All Years)')
plt.ylabel('Total Production (thousand tonnes)')
plt.xlabel('Crop')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 states by production
top_states = df.groupby('State')['Production'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 5))
top_states.plot(kind='bar', color='tomato', edgecolor='white')
plt.title('Top 10 States by Total Production')
plt.ylabel('Total Production (thousand tonnes)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Year-over-year production trend
yearly = df.groupby('Year')['Production'].sum()
plt.figure(figsize=(7, 4))
yearly.plot(kind='bar', color=['#5cb85c', '#337ab7', '#f0ad4e'], edgecolor='white')
plt.title('Total Production by Year')
plt.ylabel('Total Production (thousand tonnes)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — numeric features only
# Drop identifier columns before computing correlations
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
plt.figure(figsize=(12, 9))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix — All Numeric Features\n(Look for features highly correlated with Production)')
plt.tight_layout()
plt.show()

---
## 3. 📐 Handling Zeros & Log-Transform

### The zero-production challenge
Over 60% of rows have `Production = 0`. This happens when a district simply didn't grow that crop that year. These are **valid data points** — they shouldn't be removed. However, they do affect the distribution heavily.

### Why log-transform?
`Production` is extremely right-skewed (skewness ≈ 4.3) — a few districts produce thousands of tonnes while most produce very little. Without transformation:
- Linear models try to fit massive outliers and ignore small values
- Error metrics are dominated by large-production districts
- Tree splits get biased towards outlier thresholds

We use `log1p(x) = log(x + 1)` which safely handles zeros (`log(0+1) = 0`) and compresses the range dramatically.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Production'], bins=60, color='tomato', edgecolor='white')
axes[0].set_title(f'Production — Original\nSkewness: {df["Production"].skew():.2f}')
axes[0].set_xlabel('Production (thousand tonnes)')

log_prod = np.log1p(df['Production'])
axes[1].hist(log_prod, bins=60, color='steelblue', edgecolor='white')
axes[1].set_title(f'Production — After log1p\nSkewness: {log_prod.skew():.2f}')
axes[1].set_xlabel('log1p(Production)')

plt.suptitle('log1p transforms extreme skew into a workable distribution', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Production skewness BEFORE: {df["Production"].skew():.2f}')
print(f'Production skewness AFTER : {log_prod.skew():.2f}')

---
## 4. Feature Engineering & Preprocessing

### What we keep and what we drop
- **Drop:** `Country` (always 'India' — zero information), `Ind_adm2_ID` (geographic admin ID, redundant with Latitude/Longitude)
- **Keep:** `State_Encoded`, `District_Encoded`, `Crop_Encoded` — these are already label-encoded versions of the text columns, so we use them instead of the raw text columns to avoid needing one-hot encoding (which would create 475+ dummy columns for District alone)
- **Drop:** `State`, `District`, `Crop` raw text columns (replaced by their encoded versions)
- **Year:** Convert `'2021-22'` → `2021` (extract just the start year as a numeric feature)

### Train-Test Split (75/25)
We split before any scaling. The model learns from the 75% training set; the 25% test set is only used at evaluation time — it simulates unseen real-world data.

### StandardScaler (no leakage)
StandardScaler subtracts mean and divides by standard deviation. It's critical for SVR.
- **Fit only on training data** → learn mean/std from training set
- **Transform both train and test** using those same training statistics

Fitting on test data would be leakage — the model would indirectly see future data during training, making evaluation metrics artificially optimistic.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Feature Engineering ---
data = df.copy()

# Extract numeric year from '2021-22' → 2021
data['Year_Num'] = data['Year'].str[:4].astype(int)

# Drop non-predictive or redundant columns
data = data.drop(columns=['Country', 'Ind_adm2_ID', 'Year',
                           'State', 'District', 'Crop'])  # use encoded versions

print('Features used for modelling:')
for col in data.drop(columns=['Production']).columns:
    print(f'  {col}')

# --- Target ---
X = data.drop(columns=['Production'])
y = np.log1p(data['Production'])  # log-transform target

# --- Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# --- Scale (fit on train only) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn mean/std from train
X_test_scaled  = scaler.transform(X_test)         # apply same params to test

print(f'\nX_train: {X_train.shape}  |  X_test: {X_test.shape}')
print('✅ Scaler fitted on training data only — no leakage.')

---
## 5. Evaluation Helper

We define a helper function that computes four metrics for any model's predictions and stores the results for the final comparison table.

| Metric | Meaning | Better when |
|--------|---------|-------------|
| **R²** | % of variance in Production explained by the model | Higher (max 1.0) |
| **Adj. R²** | R² penalised for number of features — more honest | Higher |
| **RMSE** | Root mean squared error — penalises large errors heavily | Lower |
| **MAE** | Mean absolute error — treats all errors equally | Lower |

> Since Production is in log scale, RMSE = 0.5 means predictions are off by ≈ e^0.5 = 1.65x in original units.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

results = []  # accumulates all model results for Section 11

def evaluate(name, y_true, y_pred, store=True):
    n, p = len(y_true), X_test.shape[1]
    r2   = r2_score(y_true, y_pred)
    adj  = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    print(f"\n{'='*48}")
    print(f"  {name}")
    print(f"  R²        : {r2:.4f}")
    print(f"  Adj. R²   : {adj:.4f}")
    print(f"  RMSE      : {rmse:.4f}")
    print(f"  MAE       : {mae:.4f}")
    if store:
        results.append({'Model': name, 'R²': r2, 'Adj R²': adj, 'RMSE': rmse, 'MAE': mae})
    return r2

---
## 6. Baseline Models

We train four models with **default settings** first. These serve as our benchmark — any tuning or advanced model should improve upon these numbers.

- **Linear Regression** — assumes a linear relationship. Fast, but our data has complex non-linear interactions (a crop's production depends jointly on soil, weather, location, etc.).
- **Decision Tree** — learns if-else rules. Naturally handles non-linearity, but default settings often overfit (the tree grows until every leaf has just one sample).
- **Random Forest** — builds many trees independently and averages them. Overfitting is much reduced compared to a single tree.
- **SVR** — finds the best-fit hyperplane with a tolerance margin. Requires scaled features (done above). The `rbf` kernel handles non-linear patterns.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
evaluate('Linear Regression', y_test, lr_pred)

# Decision Tree (default = unlimited depth → likely to overfit)
dt_base = DecisionTreeRegressor(random_state=42)
dt_base.fit(X_train, y_train)
dt_pred = dt_base.predict(X_test)
evaluate('Decision Tree (Baseline)', y_test, dt_pred)

# Random Forest (default 100 trees)
rf_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)
rf_pred = rf_base.predict(X_test)
evaluate('Random Forest (Baseline)', y_test, rf_pred)

# SVR — uses scaled data (fitted on train only, no leakage)
svr = SVR(kernel='rbf')
svr.fit(X_train_scaled, y_train)
svr_pred = svr.predict(X_test_scaled)
evaluate('SVR (Fixed — no leakage)', y_test, svr_pred)

---
## 7. 🔧 Hyperparameter Tuning

### Why tune?
Default hyperparameters are generic starting points — they're rarely optimal for your specific dataset. Tuning can meaningfully improve R² and reduce errors.

### GridSearchCV vs RandomizedSearchCV
- **GridSearchCV** tries every combination — good when the search space is small (Decision Tree)
- **RandomizedSearchCV** randomly samples `n_iter` combinations — good when the space is large (Random Forest with 6 parameters)

Both use **5-fold cross-validation**: training data is split into 5 parts. The model trains on 4 parts and validates on 1, rotating through all 5. The average score is used to pick the best hyperparameters. This is more reliable than a single validation split.

### Key parameters for this dataset:
- `max_depth` — with 475 districts and 19 crops, deep trees can memorise patterns that don't generalise
- `min_samples_leaf` — requiring a minimum number of samples per leaf prevents overfitting on rare district-crop combos
- `n_estimators` — more trees = more stable Random Forest (at the cost of speed)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from scipy.stats import randint

# --- Decision Tree: GridSearchCV ---
# Search space is manageable (5×3×3×3 = 135 combinations)
dt_grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid={
        'max_depth': [None, 5, 10, 15, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None]
    },
    cv=5, scoring='r2', n_jobs=-1, verbose=0
)
dt_grid.fit(X_train, y_train)
best_dt = dt_grid.best_estimator_
dt_tuned_pred = best_dt.predict(X_test)

print('Best Decision Tree params:', dt_grid.best_params_)
print('Best CV R²:', round(dt_grid.best_score_, 4))
evaluate('Decision Tree (Tuned)', y_test, dt_tuned_pred)

In [ ]:
# --- Random Forest: RandomizedSearchCV ---
# Large search space → random sampling of 30 combinations is much faster
rf_random = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions={
        'n_estimators': randint(100, 400),
        'max_depth': [None, 5, 10, 15, 20, 30],
        'min_samples_split': randint(2, 15),
        'min_samples_leaf': randint(1, 8),
        'max_features': ['sqrt', 'log2', None],
        'bootstrap': [True, False]
    },
    n_iter=30, cv=5, scoring='r2', n_jobs=-1, random_state=42, verbose=0
)
rf_random.fit(X_train, y_train)
best_rf = rf_random.best_estimator_
rf_tuned_pred = best_rf.predict(X_test)

print('Best Random Forest params:', rf_random.best_params_)
print('Best CV R²:', round(rf_random.best_score_, 4))
evaluate('Random Forest (Tuned)', y_test, rf_tuned_pred)

---
## 8. ⚡ XGBoost & LightGBM — Gradient Boosting

### What is Gradient Boosting?
Unlike Random Forest (which builds trees independently and averages), gradient boosting builds trees **sequentially** — each new tree focuses on correcting the errors of all previous trees. This leads to higher accuracy but is more sensitive to hyperparameters.

### XGBoost vs LightGBM for this dataset
- **XGBoost** grows trees level-by-level (breadth-first). Solid and reliable.
- **LightGBM** grows leaf-by-leaf (depth-first on the highest-gain leaf). Faster on large datasets like this one (27K rows) and often achieves slightly better scores with the right `num_leaves`.

### Important hyperparameters here:
- `learning_rate` — lower = more conservative, needs more trees. On a dataset with complex soil+weather+geography interactions, a lower learning rate often generalises better.
- `subsample` / `colsample_bytree` — using only a fraction of rows/features per tree adds randomness and prevents overfitting
- `reg_alpha` / `reg_lambda` — L1/L2 regularisation to penalise overly complex models

In [ ]:
import xgboost as xgb

xgb_search = RandomizedSearchCV(
    xgb.XGBRegressor(random_state=42, verbosity=0, n_jobs=-1),
    param_distributions={
        'n_estimators': randint(100, 500),
        'max_depth': randint(3, 10),
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'reg_alpha': [0, 0.1, 0.5],
        'reg_lambda': [1, 1.5, 2]
    },
    n_iter=30, cv=5, scoring='r2', n_jobs=-1, random_state=42
)
xgb_search.fit(X_train, y_train)
best_xgb = xgb_search.best_estimator_
xgb_pred = best_xgb.predict(X_test)

print('Best XGBoost params:', xgb_search.best_params_)
evaluate('XGBoost (Tuned)', y_test, xgb_pred)

In [ ]:
import lightgbm as lgb

lgb_search = RandomizedSearchCV(
    lgb.LGBMRegressor(random_state=42, verbosity=-1, n_jobs=-1),
    param_distributions={
        'n_estimators': randint(100, 500),
        'max_depth': randint(3, 10),
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'num_leaves': randint(20, 100),
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'reg_alpha': [0, 0.1, 0.5],
        'reg_lambda': [1, 1.5, 2]
    },
    n_iter=30, cv=5, scoring='r2', n_jobs=-1, random_state=42
)
lgb_search.fit(X_train, y_train)
best_lgb = lgb_search.best_estimator_
lgb_pred = best_lgb.predict(X_test)

print('Best LightGBM params:', lgb_search.best_params_)
evaluate('LightGBM (Tuned)', y_test, lgb_pred)

---
## 9. 📈 Learning Curves

### What learning curves reveal
We plot training R² and validation R² as more training data is added (10% → 100%).

| Pattern | What it means | How to fix |
|---------|--------------|------------|
| Training R² >> Validation R², large gap | **Overfitting** | More data, regularisation, simpler model |
| Both scores low, converge quickly | **Underfitting** | More complex model, better features |
| Both high, small gap | **Good fit** ✅ | Nothing to do |
| Validation R² still rising at end | **Need more data** | Collect more samples |

With 27K rows, we expect the curves to converge relatively early for tree models. The shaded band = ±1 standard deviation across 5 CV folds — narrow band = stable model.

In [ ]:
from sklearn.model_selection import learning_curve

def plot_learning_curve(estimator, title, X, y, ax, color='steelblue'):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y,
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, scoring='r2', n_jobs=-1
    )
    tm, ts = train_scores.mean(axis=1), train_scores.std(axis=1)
    vm, vs = val_scores.mean(axis=1),   val_scores.std(axis=1)

    ax.plot(train_sizes, tm, 'o-', color=color,  label='Training R²')
    ax.fill_between(train_sizes, tm-ts, tm+ts, alpha=0.15, color=color)
    ax.plot(train_sizes, vm, 's--', color='tomato', label='Validation R²')
    ax.fill_between(train_sizes, vm-vs, vm+vs, alpha=0.15, color='tomato')
    ax.set_title(title)
    ax.set_xlabel('Training Samples')
    ax.set_ylabel('R² Score')
    ax.legend(loc='lower right', fontsize=8)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plot_learning_curve(best_dt,  'Decision Tree (Tuned)',  X_train, y_train, axes[0,0], '#f0ad4e')
plot_learning_curve(best_rf,  'Random Forest (Tuned)',  X_train, y_train, axes[0,1], 'steelblue')
plot_learning_curve(best_xgb, 'XGBoost (Tuned)',        X_train, y_train, axes[1,0], '#5cb85c')
plot_learning_curve(best_lgb, 'LightGBM (Tuned)',       X_train, y_train, axes[1,1], '#9b59b6')

plt.suptitle('Learning Curves\nBlue = Training R²  |  Red = Validation R²  |  Shaded = ±1 std', fontsize=13)
plt.tight_layout()
plt.show()

---
## 10. 📉 Residual Analysis

### What are residuals?
`Residual = Actual − Predicted`. If the model predicted `log(production) = 3.5` but actual was `4.0`, residual = `+0.5` (under-predicted).

### Scatter plot (Residuals vs Predicted)
- ✅ **Good:** random scatter around 0, no pattern
- ❌ **Funnel (wider at high predictions):** model struggles with high-production districts
- ❌ **Curve:** a non-linear relationship the model is missing

### Residual histogram
- ✅ **Good:** bell-shaped, centered at 0
- ❌ **Left/right skewed:** systematic over- or under-prediction

For this dataset, pay attention to residuals near `log(production) = 0` — those are the many zero-production cases, and models sometimes cluster predictions there.